In [1]:
# Week 7 - Advanced Models
# Trying Gradient Boosting (XGBoost) on top of the Week 6 enriched feature set

# Loads the enriched train/test datasets saved at the end of Week 6, rather than
# doing the entire preprocessing/feature-engineering again

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

train = pd.read_csv('/home/jorge/Documents/IDX/datasets/enriched_train.csv', low_memory=False)
test = pd.read_csv('/home/jorge/Documents/IDX/datasets/enriched_test.csv', low_memory=False)

print(train.shape)
print(test.shape)

(125549, 86)
(11631, 86)


In [3]:
# Same final feature set as the end of Week 6
numerical_features = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet',
                      'YearBuilt', 'Latitude', 'Longitude', 'AssociationFee', 'Stories',
                      'BedBathRatio', 'PropertyAge', 'BedroomAreaRatio', 'CloseMonthSin', 'CloseMonthCos']

boolean_features = ['FireplaceYN', 'NewConstructionYN', 'AttachedGarageYN', 'ViewYN', 'PoolPrivateYN']

categorical_features = ['CountyOrParish', 'DistrictGrouped']

target = 'ClosePrice'

features = numerical_features + boolean_features + categorical_features

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

In [4]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features + boolean_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(125549, 175)
(11631, 175)


In [5]:
# Baseline XGBoost - starting with conservative settings (bounded depth, modest number of trees)
# to avoid the same memory/runtime issue Random Forest caused in Week 6 with unbounded settings

from xgboost import XGBRegressor

xgb_baseline = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
xgb_baseline.fit(X_train_processed, y_train)

y_pred_xgb = xgb_baseline.predict(X_test_processed)

r2_xgb = r2_score(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
mape_xgb = mean_absolute_percentage_error(y_test, y_pred_xgb)
mdape_xgb = np.median(np.abs((y_test - y_pred_xgb) / y_test))

print(f"XGBoost (baseline) R2: {r2_xgb:.4f}")
print(f"MAE: ${mae_xgb:,.0f}")
print(f"MAPE: {mape_xgb:.2%}")
print(f"MdAPE: {mdape_xgb:.2%}")

XGBoost (baseline) R2: 0.8470
MAE: $212,658
MAPE: 16.91%
MdAPE: 12.02%


In [6]:
# Light hyperparameter tuning - checking a small set of depth/learning_rate combinations via
# cross-validation, same approach as the Decision Tree depth sweep in Week 5

# Round 3: max_depth was still climbing at 10 (the top of round 2), so this adds max_depth=12 and 15
# to check whether it keeps improving, plateaus, or starts declining the way it did for the
# Decision Tree in Week 5 (which peaked at depth 15 and dipped by depth 20)


from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

depths = [3, 5, 7, 10, 12, 15]
learning_rates = [0.05, 0.1, 0.2]

for depth in depths:
    for lr in learning_rates:
        xgb_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', XGBRegressor(n_estimators=200, max_depth=depth, learning_rate=lr, random_state=42))
        ])
        cv_scores = cross_val_score(xgb_pipeline, X_train, y_train, cv=5, scoring='r2')
        print(f"max_depth={depth}, learning_rate={lr}: CV R2={cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

max_depth=3, learning_rate=0.05: CV R2=0.7621 (+/- 0.0057)
max_depth=3, learning_rate=0.1: CV R2=0.7967 (+/- 0.0061)
max_depth=3, learning_rate=0.2: CV R2=0.8280 (+/- 0.0049)
max_depth=5, learning_rate=0.05: CV R2=0.8249 (+/- 0.0061)
max_depth=5, learning_rate=0.1: CV R2=0.8462 (+/- 0.0046)
max_depth=5, learning_rate=0.2: CV R2=0.8642 (+/- 0.0057)
max_depth=7, learning_rate=0.05: CV R2=0.8586 (+/- 0.0047)
max_depth=7, learning_rate=0.1: CV R2=0.8721 (+/- 0.0042)
max_depth=7, learning_rate=0.2: CV R2=0.8780 (+/- 0.0047)
max_depth=10, learning_rate=0.05: CV R2=0.8792 (+/- 0.0048)
max_depth=10, learning_rate=0.1: CV R2=0.8829 (+/- 0.0044)
max_depth=10, learning_rate=0.2: CV R2=0.8816 (+/- 0.0040)
max_depth=12, learning_rate=0.05: CV R2=0.8783 (+/- 0.0043)
max_depth=12, learning_rate=0.1: CV R2=0.8798 (+/- 0.0040)
max_depth=12, learning_rate=0.2: CV R2=0.8755 (+/- 0.0036)
max_depth=15, learning_rate=0.05: CV R2=0.8718 (+/- 0.0041)
max_depth=15, learning_rate=0.1: CV R2=0.8716 (+/- 0.0047)


In [7]:
# tuned model - max_depth=10, learning_rate=0.1 scored best across all 18 combinations tested
# (3 rounds of sweeping, depths 3 through 15). Deeper trees (12, 15) were checked and scored lower,
# confirming depth=10 is a genuine peak rather than an untested edge of the range.

xgb_final = XGBRegressor(n_estimators=200, max_depth=10, learning_rate=0.1, random_state=42)
xgb_final.fit(X_train_processed, y_train)

y_pred_xgb_final = xgb_final.predict(X_test_processed)

r2_xgb_final = r2_score(y_test, y_pred_xgb_final)
mae_xgb_final = mean_absolute_error(y_test, y_pred_xgb_final)
mape_xgb_final = mean_absolute_percentage_error(y_test, y_pred_xgb_final)
mdape_xgb_final = np.median(np.abs((y_test - y_pred_xgb_final) / y_test))

print(f"XGBoost (tuned) R2: {r2_xgb_final:.4f}")
print(f"MAE: ${mae_xgb_final:,.0f}")
print(f"MAPE: {mape_xgb_final:.2%}")
print(f"MdAPE: {mdape_xgb_final:.2%}")

XGBoost (tuned) R2: 0.8892
MAE: $167,090
MAPE: 12.49%
MdAPE: 8.35%


In [8]:
# Comparison against the best Week 6 model (Random Forest, 100 trees, depth 15)
# Week 6 numbers copied from the Weeks5and6/04_model_comparison.ipynb results

week7_comparison = pd.DataFrame({
    'Model': ['Random Forest (Week 6)', 'XGBoost (baseline)', 'XGBoost (tuned)'],
    'R2': [0.861864, r2_xgb, r2_xgb_final],
    'MAE': [198083.702202, mae_xgb, mae_xgb_final],
    'MAPE': [0.156331, mape_xgb, mape_xgb_final],
    'MdAPE': [0.102836, mdape_xgb, mdape_xgb_final],
})

week7_comparison['R2'] = week7_comparison['R2'].map(lambda x: f"{x:.4f}")
week7_comparison['MAE'] = week7_comparison['MAE'].map(lambda x: f"${x:,.0f}")
week7_comparison['MAPE'] = week7_comparison['MAPE'].map(lambda x: f"{x:.2%}")
week7_comparison['MdAPE'] = week7_comparison['MdAPE'].map(lambda x: f"{x:.2%}")

week7_comparison

,Model,R2,MAE,MAPE,MdAPE
0,Random Forest (Week 6),0.8619,"$198,084",15.63%,10.28%
1,XGBoost (baseline),0.8470,"$212,658",16.91%,12.02%
2,XGBoost (tuned),0.8892,"$167,090",12.49%,8.35%


## Week 7 - Advanced Models: XGBoost Behavior

| Model | R2 | MAE | MAPE | MdAPE |
|---|---|---|---|---|
| Random Forest (Week 6) | 0.8619 | `$198,084` | 15.63% | 10.28% |
| XGBoost (baseline, untuned) | 0.8470 | `$212,658` | 16.91% | 12.02% |
| XGBoost (tuned, round 1) | 0.8759 | `$185,615` | 14.41% | 10.06% |
| XGBoost (tuned, round 2 - final) | 0.8892 | `$167,090` | 12.49% | 8.35% |

